# Promoter activity KNN retrieval

This notebook trains a lightweight SeqTrainer KNN retriever that returns DNA promoters whose measured activity is closest to a requested promoter activity value.

The model indexes the numeric activity label directly. It is meant for retrieval/design workflows such as: **I need a promoter with activity 0.4; return the top 5 closest sequences.**


## 1. Imports and configuration

Use the Google Drive CSV from the project prompt or a local copy with the same schema. The expected columns are `sequence` and `label`.


In [ ]:
from pathlib import Path

import pandas as pd

from seqtrainer.models import PromoterActivityKNN

DATA_URL = "https://drive.google.com/uc?export=download&id=1Zes-RIjK1f-ctXqasH4sYALTz5vjaird"
LOCAL_DATA_PATH = Path("../data/promoter_activity_dataset.csv")
SEQUENCE_COLUMN = "sequence"
ACTIVITY_COLUMN = "label"
DEFAULT_TOP_K = 5


## 2. Load the promoter activity dataset

The loader prefers a local CSV so the notebook is reproducible offline. If the local file is missing, it tries the Google Drive download URL.


In [ ]:
if LOCAL_DATA_PATH.exists():
    df = pd.read_csv(LOCAL_DATA_PATH)
else:
    df = pd.read_csv(DATA_URL)

df.head()


## 3. Validate the expected columns


In [ ]:
required = {SEQUENCE_COLUMN, ACTIVITY_COLUMN}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required column(s): {sorted(missing)}")

df[[SEQUENCE_COLUMN, ACTIVITY_COLUMN]].describe(include="all")


## 4. Train the KNN retrieval model

For this workflow, fitting means building a nearest-neighbor index over the scalar promoter activity label.


In [ ]:
retriever = PromoterActivityKNN(n_neighbors=DEFAULT_TOP_K)
retriever.fit(
    df,
    sequence_column=SEQUENCE_COLUMN,
    value_column=ACTIVITY_COLUMN,
)

retriever


## 5. Search for promoters near a requested activity

Change `target_activity` and `top_k` to control the query and ranked-list length.


In [ ]:
target_activity = 0.4
top_k = 5

results = retriever.search(target_activity, top_k=top_k)
results


## 6. Wrap the search as a small helper


In [ ]:
def find_promoters_by_activity(target_activity: float, top_k: int = 5) -> pd.DataFrame:
    """Return ranked promoter sequences closest to a requested activity."""
    return retriever.search(target_activity, top_k=top_k)

find_promoters_by_activity(0.4, top_k=5)


## 7. Save and reload the retriever

The saved artifact contains the fitted nearest-neighbor index plus the sequence/activity records needed for search.


In [ ]:
MODEL_PATH = Path("../models/promoter_activity_knn.pkl")
retriever.save(MODEL_PATH)

loaded_retriever = PromoterActivityKNN.load(MODEL_PATH)
loaded_retriever.search(0.4, top_k=5)


## 8. Example output for downstream design tools

Most applications will only need the ranked sequence, activity label, and distance from the requested value.


In [ ]:
cols = ["rank", SEQUENCE_COLUMN, ACTIVITY_COLUMN, "query_value", "distance"]
loaded_retriever.search(0.4, top_k=5)[cols]
